# Species Mask Computation — Shared Masks for All Drugs

**One-time run** — computes per-site species-predictive bins and saves them
to `shared_masks/` for reuse across all 6 drug federated notebooks.

| Phase | Content |
|---|---|
| 1 | Per-site RF trained to classify species → saved for 08 reuse |
| 2 | Compute 3 mask strategies (union, majority, persite) |

Masks are **drug-independent** — species labels are identical across all drug CSVs.
Run once, reuse in every drug's `federated.ipynb` via `MASK_STRATEGY` config.

Compatible with Google Colab.

In [ ]:
# ── CONFIG ──
COMPUTE_DRUG = "Ciprofloxacin"  # drug with most samples per site for best RF training
MASK_TOP_K = 500

In [ ]:
!pip install maldiamrkit maldideepkit --quiet

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted')
    IN_COLAB = True
except ImportError:
    print('Running locally')
    IN_COLAB = False

In [ ]:
import warnings, json as _json
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from maldiamrkit.evaluation import stratified_species_drug_split
from maldideepkit.base.data import fit_input_transform, apply_input_transform
import joblib
warnings.filterwarnings("ignore")
SEED = 42
print("Imports ready.")

In [ ]:
# ── Paths ──
if IN_COLAB:
    DRYAD = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet")
else:
    DRYAD = Path("/media/asd/8f69beed-e984-445f-b8b3-abbb6a1a4b3f/Dryad-DataSet")

SHARED_MASKS_DIR = DRYAD / "Processed/Processing/Analysis/08-Federated-mlp-lr-rf" / "Species-Masking" / "shared_masks"
RF_MODELS_DIR = SHARED_MASKS_DIR / "rf_species_models"
SHARED_MASKS_DIR.mkdir(parents=True, exist_ok=True)
RF_MODELS_DIR.mkdir(parents=True, exist_ok=True)

SITES_PATHS = {
    "A": DRYAD / "Processed/Proc_DRIAMS-A" / COMPUTE_DRUG / "data.csv",
    "B": DRYAD / "Processed/Proc_DRIAMS-B" / COMPUTE_DRUG / "data.csv",
    "C": DRYAD / "Processed/Proc_DRIAMS-C" / COMPUTE_DRUG / "data.csv",
    "D": DRYAD / "Processed/Proc_DRIAMS-D" / COMPUTE_DRUG / "data.csv",
}
SITE_ORDER = ["A", "B", "C", "D"]
print(f"Compute drug: {COMPUTE_DRUG}")
print(f"Output dir: {SHARED_MASKS_DIR}")

In [ ]:
# ── Load data — ALL species ──
raw_data = {}
for site, path in SITES_PATHS.items():
    df = pd.read_csv(path)
    bin_cols = [c for c in df.columns if c.startswith("bin_")]
    X = df[bin_cols].to_numpy(dtype="float32")
    y = df["label"].to_numpy(dtype="int64")
    species = df["species"].values
    raw_data[site] = (X, y, species)
    n_sp = len(np.unique(species))
    print(f"  Site {site}: {len(y)} samples, {n_sp} species")

In [ ]:
# ── Per-site species-stratified 90/10 split ──
client_train = {}; client_test = {}
species_train = {}; species_test = {}

for site in SITE_ORDER:
    X, y, sp = raw_data[site]
    idx = np.arange(len(y)).reshape(-1, 1)
    idx_train, idx_val, _, _ = stratified_species_drug_split(
        idx, y, species=sp, test_size=0.10, random_state=SEED)
    idx_train = idx_train.flatten().astype(int)
    idx_val = idx_val.flatten().astype(int)
    client_train[site] = (X[idx_train], y[idx_train])
    client_test[site] = (X[idx_val], y[idx_val])
    species_train[site] = sp[idx_train]
    species_test[site] = sp[idx_val]
    print(f"  Site {site}: train={len(idx_train)} test={len(idx_val)}  species={len(np.unique(sp))}")

In [ ]:
# ── Per-site preprocessing ──
client_train_pp = {}; client_test_pp = {}
for site in SITE_ORDER:
    X_tr, y_tr = client_train[site]
    X_te, y_te = client_test[site]
    state = fit_input_transform(X_tr, "log1p+standardize")
    client_train_pp[site] = (apply_input_transform(X_tr, state), y_tr)
    client_test_pp[site] = (apply_input_transform(X_te, state), y_te)
print("Per-site preprocessing done.")

In [ ]:
# PHASE 1: Per-site RF species models
print("\n=== Phase 1: Per-Site RF Species Models ===")
site_importances = {}
site_masks_raw = {}   # top-K bin indices per site
site_n_species = {}

for site in SITE_ORDER:
    X_tr, y_tr = client_train_pp[site]
    sp_tr = species_train[site]
    n_sp = len(np.unique(sp_tr)); site_n_species[site] = n_sp

    if n_sp < 3:
        print(f"  Site {site}: only {n_sp} species — masking would be meaningless")
        site_importances[site] = np.ones(6000) / 6000  # uniform
        site_masks_raw[site] = np.array([], dtype=int)
        continue

    rf_sp = RandomForestClassifier(
        n_estimators=300, max_depth=20, min_samples_leaf=5,
        n_jobs=-1, oob_score=True, random_state=SEED)
    rf_sp.fit(X_tr, sp_tr)
    
    # Save for 08 reuse
    joblib.dump(rf_sp, str(RF_MODELS_DIR / f"site_{site}_rf_species.joblib"))

    site_importances[site] = rf_sp.feature_importances_
    top_k = np.argsort(rf_sp.feature_importances_)[-MASK_TOP_K:]
    top_k.sort()
    site_masks_raw[site] = top_k
    explained = rf_sp.feature_importances_[top_k].sum()
    print(f"  Site {site}: {n_sp} species, OOB acc={rf_sp.oob_score_:.4f}, "
          f"top-{MASK_TOP_K} bins explain {explained:.1%} of species importance")

print("\nPhase 1 done. RF models saved to rf_species_models/.")

In [ ]:
# PHASE 2: Compute mask strategies
print("\n=== Phase 2: Mask Computation ===")

# ── Union mask: any bin flagged by any site ──
all_site_bins = [site_masks_raw[s] for s in SITE_ORDER if len(site_masks_raw[s]) > 0]
union_mask = np.unique(np.concatenate(all_site_bins)) if all_site_bins else np.array([], dtype=int)
print(f"  Union mask: {len(union_mask)} bins")

# ── Majority mask: bin flagged by >= 2 sites ──
bin_counts = {}
for bins in all_site_bins:
    for b in bins:
        bin_counts[b] = bin_counts.get(b, 0) + 1
majority_mask = np.array([b for b, c in bin_counts.items() if c >= 2], dtype=int)
majority_mask.sort()
print(f"  Majority mask: {len(majority_mask)} bins (flagged by >=2 sites)")

# ── Persite mask: each site uses its own top-K ──
print(f"  Persite: {MASK_TOP_K} bins per site")

# Save all masks
np.save(str(SHARED_MASKS_DIR / "union_mask.npy"), union_mask)
np.save(str(SHARED_MASKS_DIR / "majority_mask.npy"), majority_mask)
for site in SITE_ORDER:
    np.save(str(SHARED_MASKS_DIR / f"persite_site_{site}_mask.npy"), site_masks_raw[site])

# ── Pairwise mask overlaps ──
print("\n  Pairwise mask overlaps (% shared bins):")
sites_with_mask = [s for s in SITE_ORDER if len(site_masks_raw[s]) > 0]
for i, sa in enumerate(sites_with_mask):
    for j, sb in enumerate(sites_with_mask):
        if j <= i: continue
        overlap = len(set(site_masks_raw[sa]) & set(site_masks_raw[sb]))
        pct = overlap / MASK_TOP_K * 100
        print(f"    {sa}∩{sb}: {overlap}/{MASK_TOP_K} ({pct:.0f}%)")

In [ ]:
# ── Save metadata ──
metadata = {
    "compute_drug": COMPUTE_DRUG,
    "mask_top_k": MASK_TOP_K,
    "computed_at": datetime.now().isoformat(),
    "site_n_species": site_n_species,
    "mask_sizes": {
        "union": int(len(union_mask)),
        "majority": int(len(majority_mask)),
        "persite": {s: int(len(site_masks_raw[s])) for s in SITE_ORDER}
    }
}
with open(str(SHARED_MASKS_DIR / "metadata.json"), "w") as f:
    _json.dump(metadata, f, indent=2)

print(f"\nMetadata saved to {SHARED_MASKS_DIR / 'metadata.json'}")
for k, v in metadata["mask_sizes"].items():
    print(f"  {k}: {v}")

---
**Done.** Species masks computed and saved.

| Output | Location |
|---|---|
| RF species models | `shared_masks/rf_species_models/site_{A,B,C,D}_rf_species.joblib` |
| Union mask | `shared_masks/union_mask.npy` |
| Majority mask | `shared_masks/majority_mask.npy` |
| Persite masks | `shared_masks/persite_site_{A,B,C,D}_mask.npy` |
| Metadata | `shared_masks/metadata.json` |

To use in per-drug federated notebooks, set:

```python
USE_SPECIES_MASKING = True
MASK_STRATEGY = "majority"  # "none" | "union" | "majority" | "persite"
SHARED_MASKS = DRYAD / "Processed/Processing/Analysis/08-Federated-mlp-lr-rf" / "Species-Masking" / "shared_masks"
```